Same evaluation logic(timeseries, event plots, scatter plots, confusion matrix, and metrics) as `determinist_fixed_LT.py` but for one FOEN forcing configuation coverage period at a time. Outputs are saved to `outputs/OFEV_determinist/<OFEV_model>/`.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LEAD_TIMES_EVAL = [6, 12, 24, 36, 48]
FLOOD_THR = 400

# Define paths
ROOT = Path("..").resolve()
BASE_DIR = ROOT / "dp_arve"
DATA_DIR = BASE_DIR / "data/Data Fornisseurs"
if not DATA_DIR.exists():
    DATA_DIR = ROOT / "data/Data Fornisseurs"
OUT_BASE = ROOT / "outputs" / "OFEV_determinist"
OUT_BASE.mkdir(parents=True, exist_ok=True)

DET_CSV = ROOT / "outputs" / "OFEV_probabilistic" / "station2170_deterministic_with_q75.csv"
OBS_PATH = DATA_DIR / "ARVE" / "2170_Abfluss_10-Min-Mittel_1999-01-01_2024-12-31.csv"
ML_JSON = DATA_DIR / "Hydrique_model" / "Archive prévisions Hydrique ML Arve-Bout du Monde.json"
HYD_JSON = DATA_DIR / "Hydrique_model" / "Archive prévisions Hydrique hydrique-curve Arve-Bout du Monde.json"
CNR_CSV = DATA_DIR / "SIG-CNR_model" / "Previsions_CNR_20_25.csv"

# Define metrics functions
def nse(obs, sim):
    den = np.sum((obs - np.mean(obs)) ** 2)
    return np.nan if den == 0 else 1 - np.sum((obs - sim) ** 2) / den


def kge(obs, sim):
    if len(obs) < 2:
        return np.nan
    r = np.corrcoef(obs, sim)[0, 1]
    if np.isnan(r) or np.std(obs) == 0 or np.mean(obs) == 0:
        return np.nan
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)
    return 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)


def peak_error(obs, sim):
    m = np.max(obs)
    return np.nan if m == 0 else (np.max(sim) - m) / m * 100


def peak_timing(obs, sim, index):
    return (index[np.argmax(sim)] - index[np.argmax(obs)]).total_seconds() / 3600


def rmse(obs, sim):
    return np.sqrt(np.mean((obs - sim) ** 2))


def relative_volume_error(obs, sim):
    den = np.sum(obs)
    return np.nan if den == 0 else (np.sum(sim) - den) / den * 100


def mape_high_flows(obs, sim, thr):
    mask = obs > thr
    if mask.sum() == 0:
        return np.nan
    den = obs[mask]
    den = np.where(den == 0, np.nan, den)
    return 100 * np.nanmean(np.abs((obs[mask] - sim[mask]) / den))


def get_flood_events(series, thr, gap_hours=12):
    mask = series > thr
    groups = (mask != mask.shift()).cumsum()
    raw = [(g.index.min(), g.index.max()) for _, g in series[mask].groupby(groups)]
    if not raw:
        return []
    merged = [raw[0]]
    for start, end in raw[1:]:
        ps, pe = merged[-1]
        if (start - pe).total_seconds() / 3600 <= gap_hours:
            merged[-1] = (ps, end)
        else:
            merged.append((start, end))
    return merged


def event_confusion_matrix(df, thr):
    obs_events = get_flood_events(df["Q_obs"], thr)
    sim_events = get_flood_events(df.iloc[:, 1], thr)
    tp = fp = fn = 0
    for start, end in obs_events:
        tp += int(df.loc[start:end].iloc[:, 1].max() > thr)
        fn += int(df.loc[start:end].iloc[:, 1].max() <= thr)
    for start, end in sim_events:
        fp += int(df.loc[start:end]["Q_obs"].max() <= thr)
    return tp, fp, fn


def event_scores(tp, fp, fn):
    pod = tp / (tp + fn) if (tp + fn) else np.nan
    far = fp / (tp + fp) if (tp + fp) else np.nan
    csi = tp / (tp + fp + fn) if (tp + fp + fn) else np.nan
    return pod, far, csi


def peak_timing_events(df, thr):
    errs = []
    for start, end in get_flood_events(df["Q_obs"], thr):
        sub = df.loc[start:end]
        if len(sub) < 3 or sub.iloc[:, 1].isna().all():
            continue
        errs.append((sub.index[np.argmax(sub.iloc[:, 1].values)] - sub.index[np.argmax(sub["Q_obs"].values)]).total_seconds() / 3600)
    return np.mean(errs) if errs else np.nan


# Plotting function for confusion matrix
def plot_confusion_matrix(tp, fp, fn, title, out_file):
    cm = np.array([[0, fp], [fn, tp]])
    total = tp + fp + fn
    cmn = cm / total if total > 0 else cm
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            txt = "NA" if (i == 0 and j == 0) else f"{cm[i, j]}\n({cmn[i, j]:.2f})"
            ax.text(j, i, txt, ha="center", va="center")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["No Flood", "Flood"]); ax.set_yticklabels(["No Flood", "Flood"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Observed"); ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.65)
    plt.tight_layout(); plt.savefig(out_file, dpi=220); plt.close()


# Data loading functions for each source, with cleaning and conversion to common format
def load_obs_hourly(path):
    obs = pd.read_csv(path, sep=";", encoding="cp1252", skiprows=8)
    obs.columns = obs.columns.str.strip()
    obs = obs.rename(columns={"Zeitstempel": "datetime", "Wert": "Q_obs"})
    obs = obs[["datetime", "Q_obs"]]
    obs["datetime"] = pd.to_datetime(obs["datetime"], errors="coerce")
    obs["Q_obs"] = pd.to_numeric(obs["Q_obs"], errors="coerce")
    obs = obs.dropna().drop_duplicates(subset=["datetime"]).sort_values("datetime")
    return obs.set_index("datetime").resample("1h").mean()


def load_hydrique_json(path, q_col):
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    rec = [(ft, vt, q) for ft, ts in d["ForecastFirstDate2Timeseries"].items() for vt, q in ts.items()]
    df = pd.DataFrame(rec, columns=["forecast_time", "valid_time", q_col])
    df["forecast_time"] = pd.to_datetime(df["forecast_time"], errors="coerce")
    df["valid_time"] = pd.to_datetime(df["valid_time"], errors="coerce")
    df[q_col] = pd.to_numeric(df[q_col], errors="coerce")
    df["lead_time_h"] = (df["valid_time"] - df["forecast_time"]).dt.total_seconds() / 3600
    return df.set_index("valid_time")


def _norm_col(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


def load_cnr_csv(path):
    df = pd.read_csv(path)
    cmap = {c: _norm_col(c) for c in df.columns}
    inv = {v: k for k, v in cmap.items()}

    prev_col = inv.get("datetime_prev")
    valid_col = inv.get("date_prevision")
    q_col = inv.get("q") or inv.get("debit") or inv.get("discharge")

    if prev_col is None or valid_col is None or q_col is None:
        raise ValueError(f"Unexpected CNR columns: {list(df.columns)}")

    out = pd.DataFrame({
        "forecast_time": pd.to_datetime(df[prev_col], errors="coerce"),
        "valid_time": pd.to_datetime(df[valid_col], errors="coerce"),
        "Q_cnr": pd.to_numeric(df[q_col], errors="coerce"),
    })
    out = out.dropna(subset=["valid_time", "Q_cnr"])
    out["lead_time_h"] = (out["valid_time"] - out["forecast_time"]).dt.total_seconds() / 3600
    return out.set_index("valid_time")


def load_cnr_xlsx(path):
    xls = pd.ExcelFile(path)
    best = None
    for sh in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sh)
        if df.empty:
            continue
        cmap = {c: _norm_col(c) for c in df.columns}
        if {"datetime_prev", "date_prevision"}.issubset(set(cmap.values())):
            inv = {v: k for k, v in cmap.items()}
            out = pd.DataFrame({
                "forecast_time": pd.to_datetime(df[inv["datetime_prev"]], errors="coerce"),
                "valid_time": pd.to_datetime(df[inv["date_prevision"]], errors="coerce"),
                "Q_cnr": pd.to_numeric(df[inv.get("q", inv.get("debit", list(df.columns)[-1]))], errors="coerce"),
            })
        else:
            dt_candidates = [c for c, lc in cmap.items() if any(k in lc for k in ["date", "time", "timestamp", "prevision"])]
            q_candidates = [c for c, lc in cmap.items() if any(k in lc for k in ["debit", "flow", "discharge", "m3", "_q", "q_"]) or lc == "q"]
            if not dt_candidates:
                continue
            dt_col = max(dt_candidates, key=lambda c: pd.to_datetime(df[c], errors="coerce").notna().mean())
            q_col = max((q_candidates or list(df.columns)), key=lambda c: pd.to_numeric(df[c], errors="coerce").notna().mean())
            out = pd.DataFrame({
                "valid_time": pd.to_datetime(df[dt_col], errors="coerce"),
                "Q_cnr": pd.to_numeric(df[q_col], errors="coerce"),
            })
            out["forecast_time"] = out["valid_time"]

        out = out.dropna(subset=["valid_time", "Q_cnr"])
        out["lead_time_h"] = (out["valid_time"] - out["forecast_time"]).dt.total_seconds() / 3600
        out = out.set_index("valid_time")
        if best is None or len(out) > len(best):
            best = out

    return best if best is not None else pd.DataFrame(columns=["forecast_time", "Q_cnr", "lead_time_h"])


def clean_q_frame(df, col):
    out = df[[col]].copy()
    out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(subset=[col])
    out = out[out[col] >= 0]
    return out.sort_index()


def safe_name(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s)).strip("_")

# Infer problems for missing metrics 
def infer_problem_reasons(obs_v, sim_v, tp, fp, fn, nse_v, kge_v, rmse_v, pod, far, csi, has_enough_high):
    reasons = []

    if pd.isna(nse_v):
        den = np.sum((obs_v - np.mean(obs_v)) ** 2)
        reasons.append("ZERO_OBS_VARIANCE" if den == 0 else "NSE_UNDEFINED")

    if pd.isna(kge_v):
        if len(obs_v) < 2:
            reasons.append("TOO_FEW_POINTS")
        if np.std(obs_v) == 0:
            reasons.append("ZERO_OBS_VARIANCE")
        if np.mean(obs_v) == 0:
            reasons.append("ZERO_OBS_MEAN")
        if len(obs_v) >= 2 and np.isnan(np.corrcoef(obs_v, sim_v)[0, 1]):
            reasons.append("UNDEFINED_CORRELATION")
        if not reasons:
            reasons.append("KGE_UNDEFINED")

    if pd.isna(rmse_v):
        reasons.append("RMSE_UNDEFINED")

    if pd.isna(pod) and (tp + fn) == 0:
        reasons.append("NO_OBS_EVENTS")
    if pd.isna(far) and (tp + fp) == 0:
        reasons.append("NO_SIM_EVENTS")
    if pd.isna(csi) and (tp + fp + fn) == 0:
        reasons.append("NO_EVENTS_ANY")

    if not has_enough_high:
        reasons.append("TOO_FEW_HIGH_FLOW_POINTS")

    return "|".join(dict.fromkeys(reasons))

# Load inputs 
obs_h = load_obs_hourly(OBS_PATH)
ml = load_hydrique_json(ML_JSON, "Q_ml")
hyd = load_hydrique_json(HYD_JSON, "Q_hyd")
cnr = load_cnr_csv(CNR_CSV)

ofev_det = pd.read_csv(DET_CSV, parse_dates=["issue_time", "valid_time"])
ofev_det["lead_time_h"] = pd.to_numeric(ofev_det["lead_time_h"], errors="coerce")
ofev_det["Q"] = pd.to_numeric(ofev_det["Q"], errors="coerce")
ofev_det = ofev_det.dropna(subset=["model", "valid_time", "Q", "lead_time_h"])
ofev_det = ofev_det[ofev_det["Q"] >= 0]
ofev_det = ofev_det.set_index("valid_time")

all_rows = []

# Analyze each OFEV model separatelyand compare to all other sources on that common period
for ofev_model in sorted(ofev_det["model"].astype(str).unique()):
    model_dir = OUT_BASE / safe_name(ofev_model)
    cm_dir = model_dir / "Confusion Matrix"
    model_dir.mkdir(parents=True, exist_ok=True)
    cm_dir.mkdir(parents=True, exist_ok=True)

    event_totals = {"OFEV": np.array([0, 0, 0]), "Hydrique_ML": np.array([0, 0, 0]), "Hydrique_physique": np.array([0, 0, 0]), "SIG_CNR": np.array([0, 0, 0])}
    model_metrics = []

    for lt in LEAD_TIMES_EVAL:
        out_lt = model_dir / f"leadtime_{lt}h"
        out_lt.mkdir(parents=True, exist_ok=True)

        ofev_lt = ofev_det[(ofev_det["model"].astype(str) == ofev_model) & (ofev_det["lead_time_h"].round().astype(int) == lt)][["Q"]].rename(columns={"Q": "Q_ofev"})
        ofev_lt = ofev_lt.groupby(ofev_lt.index).median().sort_index()
        ofev_lt = clean_q_frame(ofev_lt, "Q_ofev")
        if ofev_lt.empty:
            continue

        # Restrict all comparisons strictly to this OFEV model coverage
        t0, t1 = ofev_lt.index.min(), ofev_lt.index.max()
        obs_plot = clean_q_frame(obs_h.loc[t0:t1], "Q_obs")
        ml_plot = clean_q_frame(ml[ml["lead_time_h"].round() == lt][["Q_ml"]].loc[t0:t1], "Q_ml")
        hyd_plot = clean_q_frame(hyd[hyd["lead_time_h"].round() == lt][["Q_hyd"]].loc[t0:t1], "Q_hyd")
        cnr_plot = clean_q_frame(cnr[cnr["lead_time_h"].round() == lt][["Q_cnr"]].loc[t0:t1], "Q_cnr")
        if obs_plot.empty:
            continue

        # Timeseries plot on OFEV-model-specific period.
        plt.figure(figsize=(16, 6))
        plt.plot(obs_plot.index, obs_plot["Q_obs"], color="black", label="Observed")
        if not ml_plot.empty: plt.plot(ml_plot.index, ml_plot["Q_ml"], label="Hydrique ML")
        if not hyd_plot.empty: plt.plot(hyd_plot.index, hyd_plot["Q_hyd"], label="Hydrique Physique")
        if not cnr_plot.empty: plt.plot(cnr_plot.index, cnr_plot["Q_cnr"], label="SIG-CNR")
        plt.plot(ofev_lt.index, ofev_lt["Q_ofev"], label=f"OFEV {ofev_model}")
        plt.xlim(t0, t1); plt.legend(); plt.title(f"{ofev_model} - Timeseries LT {lt}h")
        plt.tight_layout(); plt.savefig(out_lt / f"timeseries_LT{lt}.png", dpi=200); plt.close()

        # Event windows plot (OBS events only, expanded +-2 days)
        for i, (start, end) in enumerate(get_flood_events(obs_plot["Q_obs"], FLOOD_THR)):
            sw, ew = start - pd.Timedelta(days=2), end + pd.Timedelta(days=2)
            fig, ax = plt.subplots(figsize=(16, 6))
            ax.plot(obs_plot.loc[sw:ew].index, obs_plot.loc[sw:ew, "Q_obs"], color="black", label="Observed")
            if not ml_plot.empty: ax.plot(ml_plot.loc[sw:ew].index, ml_plot.loc[sw:ew, "Q_ml"], label="Hydrique ML")
            if not hyd_plot.empty: ax.plot(hyd_plot.loc[sw:ew].index, hyd_plot.loc[sw:ew, "Q_hyd"], label="Hydrique Physique")
            if not cnr_plot.empty: ax.plot(cnr_plot.loc[sw:ew].index, cnr_plot.loc[sw:ew, "Q_cnr"], label="SIG-CNR")
            ax.plot(ofev_lt.loc[sw:ew].index, ofev_lt.loc[sw:ew, "Q_ofev"], label=f"OFEV {ofev_model}")
            ax.set_xlim(sw, ew)
            ax.set_title(f"{ofev_model} flood {sw.strftime('%d/%m')} - {ew.strftime('%d/%m/%Y')} (LT {lt}h)")
            ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%d/%m\n%H:%M"))
            ax.legend(); plt.tight_layout()
            plt.savefig(out_lt / f"flood_{sw.strftime('%Y%m%d')}_{ew.strftime('%Y%m%d')}_LT{lt}_{i+1:02d}.png", dpi=200)
            plt.close()

        # Metrics for each compared model on the same OFEV-model period
        for name, s in [("OFEV", ofev_lt), ("Hydrique_ML", ml_plot), ("Hydrique_physique", hyd_plot), ("SIG_CNR", cnr_plot)]:
            df = obs_plot.join(s, how="inner").dropna()
            df = df[(df["Q_obs"] >= 0) & (df.iloc[:, 1] >= 0)]
            if len(df) < 10:
                continue

            obs_v, sim_v = df["Q_obs"].values, df.iloc[:, 1].values
            df_high = df[df["Q_obs"] > FLOOD_THR]
            obs_hf, sim_hf = (df_high["Q_obs"].values, df_high.iloc[:, 1].values) if len(df_high) > 5 else (None, None)
            has_enough_high = len(df_high) > 5

            tp, fp, fn = event_confusion_matrix(df, FLOOD_THR)
            pod, far, csi = event_scores(tp, fp, fn)
            event_totals[name] += np.array([tp, fp, fn])
            plot_confusion_matrix(tp, fp, fn, f"{ofev_model} | {name} - LT{lt}", cm_dir / f"CM_{name}_LT{lt}.png")

            reqs, tps, rers = [], [], []
            for start, end in get_flood_events(df["Q_obs"], FLOOD_THR):
                de = df.loc[start - pd.Timedelta(days=2): end + pd.Timedelta(days=2)]
                de = de.dropna()
                de = de[(de["Q_obs"] >= 0) & (de.iloc[:, 1] >= 0)]
                if len(de) < 5:
                    continue
                reqs.append(peak_error(de["Q_obs"].values, de.iloc[:, 1].values))
                tps.append(peak_timing(de["Q_obs"].values, de.iloc[:, 1].values, de.index))
                rers.append(relative_volume_error(de["Q_obs"].values, de.iloc[:, 1].values))

            nse_v = nse(obs_v, sim_v)
            kge_v = kge(obs_v, sim_v)
            rmse_v = rmse(obs_v, sim_v)
            mape_v = mape_high_flows(obs_v, sim_v, FLOOD_THR)
            nse_high_v = nse(obs_hf, sim_hf) if obs_hf is not None else np.nan
            kge_high_v = kge(obs_hf, sim_hf) if obs_hf is not None else np.nan
            rmse_high_v = rmse(obs_hf, sim_hf) if obs_hf is not None else np.nan
            req_high_v = peak_error(obs_hf, sim_hf) if obs_hf is not None else np.nan
            tp_high_v = peak_timing_events(df, FLOOD_THR)
            rer_high_v = relative_volume_error(obs_hf, sim_hf) if obs_hf is not None else np.nan
            problem_reason = infer_problem_reasons(obs_v, sim_v, tp, fp, fn, nse_v, kge_v, rmse_v, pod, far, csi, has_enough_high)
            coverage_start = df.index.min().isoformat()
            coverage_end = df.index.max().isoformat()

            model_metrics.append({
                "ofev_model": ofev_model,
                "lead_time": lt,
                "model": name,
                "N_rows": len(df),
                "coverage_start": coverage_start,
                "coverage_end": coverage_end,
                "NSE": nse_v,
                "KGE": kge_v,
                "RMSE": rmse_v,
                "REQ_%": np.median(reqs) if reqs else np.nan,
                "TP_h": np.median(tps) if tps else np.nan,
                "RER_%": np.median(rers) if rers else np.nan,
                "MAPE_high_%": mape_v,
                "POD": pod,
                "FAR": far,
                "CSI": csi,
                "NSE_high": nse_high_v,
                "KGE_high": kge_high_v,
                "RMSE_high": rmse_high_v,
                "REQ_high_%": req_high_v,
                "TP_high_h": tp_high_v,
                "RER_high_%": rer_high_v,
                "problem_reason": problem_reason,
            })

            # Scatter plots of sim vs obs
            plt.figure(figsize=(5, 5))
            plt.scatter(obs_v, sim_v, s=5, alpha=0.3)
            mmax = max(np.nanmax(obs_v), np.nanmax(sim_v))
            plt.plot([0, mmax], [0, mmax], "k--")
            plt.xlabel("Observed"); plt.ylabel(name)
            plt.title(f"{ofev_model} - {name} LT{lt}")
            plt.tight_layout(); plt.savefig(out_lt / f"scatter_{name}_LT{lt}.png", dpi=200); plt.close()

    for name, vals in event_totals.items():
        tp, fp, fn = vals
        plot_confusion_matrix(tp, fp, fn, f"{ofev_model} | {name} - ALL_LT", cm_dir / f"CM_{name}_ALL_LT.png")

    model_metrics = pd.DataFrame(model_metrics)
    model_metrics.to_csv(model_dir / "metrics_all_models.csv", index=False)
    all_rows.append(model_metrics)

summary = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
summary.to_csv(OUT_BASE / "metrics_all_models_summary.csv", index=False)